# RipeSense: train on public datasets (no API keys)

This notebook downloads free public datasets, merges them into one training set, and trains the
models the RipeSense app already knows how to load.

| Part | Data | Output |
|---|---|---|
| Camera | 20 image datasets (Mendeley, Hugging Face, Kaggle, Laboro, Figshare) | `fruit_classifier.onnx` + `fruit_classifier.onnx.labels.json` |
| Gas | 10 gas-sensor datasets (Zenodo, Figshare, UCI, Mendeley) | `public_gas_rows.csv`, `ethylene_curve.json` |
| O2 / respiration | published measurements (no public raw O2 logs exist) | `respiration_reference.csv` + chamber estimate |

**How to run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.
Expect about 40–70 minutes total, most of it downloads. At the end, download **`ripesense_models.zip`**
from the Files panel and copy the files into `~/ripesense/models/` on the Pi.

Nothing here needs a login: Mendeley, Zenodo, Figshare, UCI and Hugging Face allow anonymous downloads,
and `kagglehub` downloads public Kaggle datasets without an account. If a source is down, its cell
logs the error and the rest keeps going.

In [ ]:
# ── Settings ───────────────────────────────────────────────────────────────────
WORK = "/content/rs"            # everything is downloaded and built here
MAX_FILE_GB = 5.0               # skip any single download bigger than this
PER_CLASS_CAP = 700             # max images per (dataset, fruit, stage); stops one dataset dominating
MIN_CLASS = 25                  # drop a fruit__stage class with fewer images than this
EPOCHS = 14
BATCH = 64
QUICK = False                   # True: tiny caps and 1 epoch, to check the notebook end to end in ~10 min
ETHYLENE_EXTRAS = False         # True: also fetch 4 large UCI ethylene time-series sets (~5 GB, not fruit)

if QUICK:
    PER_CLASS_CAP, EPOCHS, MAX_FILE_GB = 60, 1, 1.0

In [ ]:
# ── Install ────────────────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub", "huggingface_hub", "datasets",
                "remotezip", "rarfile", "openpyxl", "onnx", "onnxruntime", "pyyaml"], check=False)
subprocess.run("apt-get -qq install -y unrar > /dev/null 2>&1", shell=True, check=False)

In [ ]:
# ── Helpers: download, extract, label from folder names ────────────────────────
import concurrent.futures as cf, hashlib, io, json, os, random, re, shutil, tarfile, time, traceback, zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import requests
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None
W = Path(WORK); (W / "raw").mkdir(parents=True, exist_ok=True); (W / "unified").mkdir(exist_ok=True)
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
ARCH_EXT = (".zip", ".rar", ".tar", ".tar.gz", ".tgz")
S = requests.Session(); S.headers["User-Agent"] = "ripesense-training/1.0"
LOG = []                        # (dataset, status, detail) for the summary table


def log(ds, status, detail=""):
    LOG.append((ds, status, str(detail)[:160]))
    print(f"[{ds}] {status} {detail}")


def download(url, dest: Path, size_hint=None, tries=3):
    """Stream a URL to disk with retries. Skips files bigger than MAX_FILE_GB."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0 and (size_hint is None or dest.stat().st_size == size_hint):
        return dest
    if size_hint and size_hint > MAX_FILE_GB * 1e9:
        raise RuntimeError(f"{dest.name} is {size_hint / 1e9:.1f} GB > MAX_FILE_GB")
    for k in range(tries):
        try:
            with S.get(url, stream=True, timeout=60, allow_redirects=True) as r:
                r.raise_for_status()
                n = int(r.headers.get("content-length") or 0)
                if n > MAX_FILE_GB * 1e9:
                    raise RuntimeError(f"{dest.name} is {n / 1e9:.1f} GB > MAX_FILE_GB")
                tmp = dest.with_suffix(dest.suffix + ".part")
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(1 << 22):
                        f.write(chunk)
                tmp.rename(dest)
                return dest
        except RuntimeError:
            raise
        except Exception as e:
            if k == tries - 1:
                raise
            time.sleep(3 * (k + 1))


def extract_all(root: Path, depth=2):
    """Extract every archive under root (and archives inside those), then delete the archive."""
    for _ in range(depth):
        found = False
        for p in list(root.rglob("*")):
            name = p.name.lower()
            if not p.is_file() or not name.endswith(ARCH_EXT):
                continue
            found = True
            out = p.parent / (p.name.split(".")[0] + "_x")
            try:
                if name.endswith(".zip"):
                    with zipfile.ZipFile(p) as z:
                        z.extractall(out)
                elif name.endswith(".rar"):
                    import rarfile
                    with rarfile.RarFile(p) as z:
                        z.extractall(out)
                else:
                    with tarfile.open(p) as t:
                        t.extractall(out)
                p.unlink()
            except Exception as e:
                print("   could not extract", p.name, e)
        if not found:
            break


# --- Mendeley Data public API (no key) ---
MD = "https://data.mendeley.com/public-api/datasets"


def mendeley_files(ds_id, version=None):
    """All files of a Mendeley dataset as (relative_path, url, size), walking its folders."""
    if version is None:
        version = S.get(f"{MD}/{ds_id}", timeout=30).json().get("version", 1)
    folders = {}
    try:
        for f in S.get(f"{MD}/{ds_id}/folders/{version}", timeout=30).json():
            folders[f["id"]] = (f["name"], f.get("parent_id"))
    except Exception:
        pass

    def path_of(fid):
        parts = []
        while fid and fid in folders:
            name, fid = folders[fid]
            parts.append(name)
        return "/".join(reversed(parts))

    out = []
    for fid in ["root"] + list(folders):
        try:
            files = S.get(f"{MD}/{ds_id}/files", params={"folder_id": fid, "version": version}, timeout=60).json()
        except Exception:
            continue
        for f in files if isinstance(files, list) else []:
            rel = (path_of(fid) + "/" if fid != "root" else "") + f["filename"]
            url = f.get("download_url") or (f.get("content_details") or {}).get("download_url")
            if url:
                out.append((rel, url, f.get("size")))
    return out


def fetch_mendeley(name, ds_id, version=None, include=None, exclude=None, max_files=6000):
    """include/exclude: regexes on the file path, e.g. exclude='augment' to skip augmented copies."""
    dest = W / "raw" / name
    files = mendeley_files(ds_id, version)
    keep = []
    for rel, url, size in files:
        low = rel.lower()
        if include and not re.search(include, low):
            continue
        if exclude and re.search(exclude, low):
            continue
        if not (low.endswith(ARCH_EXT) or Path(low).suffix in IMG_EXT | {".yaml", ".yml", ".txt", ".json",
                                                                         ".csv", ".xlsx", ".xls", ".dat"}):
            continue
        if "/.git/" in "/" + low or (low.startswith("yolov5/") and not low.endswith((".yaml", ".yml"))):
            continue
        keep.append((rel, url, size))
    if not keep:
        raise RuntimeError(f"no usable files in Mendeley {ds_id} ({len(files)} listed)")
    if len(keep) > max_files:              # thousands of loose images: take a random subset (labels kept)
        lab = [k for k in keep if not Path(k[0]).suffix.lower() in IMG_EXT]
        img = [k for k in keep if Path(k[0]).suffix.lower() in IMG_EXT]
        random.Random(0).shuffle(img)
        keep = lab + img[:max_files]
    total = sum(s or 0 for _, _, s in keep)
    print(f"   {name}: {len(keep)} files, {total / 1e9:.2f} GB")
    with cf.ThreadPoolExecutor(16 if len(keep) > 20 else 4) as ex:
        futs = [ex.submit(download, url, dest / rel, size) for rel, url, size in keep
                if not (size and size > MAX_FILE_GB * 1e9)]
        for fu in cf.as_completed(futs):
            try:
                fu.result()
            except Exception as e:
                print("   skip:", e)
    extract_all(dest)
    return dest


def fetch_url(name, url, filename):
    dest = W / "raw" / name
    download(url, dest / filename)
    extract_all(dest)
    return dest


def fetch_kaggle(name, slug):
    import kagglehub
    p = Path(kagglehub.dataset_download(slug))
    dest = W / "raw" / name
    if not dest.exists():
        dest.symlink_to(p, target_is_directory=True)
    return dest


def fetch_hf_parquet(name, repo, config=None):
    """Hugging Face image datasets: write every image into label-named folders."""
    from datasets import load_dataset
    dest = W / "raw" / name
    if dest.exists() and any(dest.rglob("*.jpg")):
        return dest
    try:
        ds = load_dataset(repo, config, split="train")
    except Exception as e:            # repo holds a plain zip of folders: download and unpack it
        print("   load_dataset failed, downloading files instead:", type(e).__name__)
        from huggingface_hub import snapshot_download
        snapshot_download(repo, repo_type="dataset", local_dir=str(dest))
        extract_all(dest)
        return dest
    feats = ds.features
    lab_names = feats["label"].names if "label" in feats and hasattr(feats["label"], "names") else None
    n = Counter()
    for i, ex in enumerate(ds):
        lab = ex.get("label")
        lab = lab_names[lab] if lab_names is not None and isinstance(lab, int) else str(lab)
        fruit = str(ex.get("fruit_type") or ex.get("fruit") or "")
        key = (fruit, lab)
        if n[key] >= PER_CLASS_CAP * 2:
            continue
        n[key] += 1
        d = dest / lab / (fruit or "unknown")
        d.mkdir(parents=True, exist_ok=True)
        img = ex["image"].convert("RGB")
        img.thumbnail((400, 400))
        img.save(d / f"{i}.jpg", quality=90)
    return dest


def fetch_remotezip(name, url, member_re, max_members=None, exclude_re=None):
    """Pull only some members out of a huge zip with HTTP range requests (no full download)."""
    from remotezip import RemoteZip
    dest = W / "raw" / name
    dest.mkdir(parents=True, exist_ok=True)
    rx = re.compile(member_re, re.I)
    with RemoteZip(url) as z:
        names = [n for n in z.namelist() if rx.search(n) and not n.endswith("/")
                 and not (exclude_re and re.search(exclude_re, n, re.I))]
        if max_members:
            random.Random(0).shuffle(names)
            by_folder = defaultdict(list)
            for n in names:
                by_folder[str(Path(n).parent)].append(n)
            names = [n for lst in by_folder.values() for n in lst[:max_members]]
        print(f"   {name}: {len(names)} members")
        for n in names:
            out = dest / n
            if out.exists():
                continue
            out.parent.mkdir(parents=True, exist_ok=True)
            out.write_bytes(z.read(n))
    return dest


# --- Labels: folder and file names → (fruit, stage) ---
STAGE_RULES = [  # checked in this order; first match wins
    ("ripe",     r"not[\s_-]*spoil|non[\s_-]*spoil|unspoil|not[\s_-]*rotten|non[\s_-]*rotten"),
    ("skip",     r"formalin|mixed|carbide|artificial"),
    ("overripe", r"over[\s_-]*ripe|overripen|too[\s_-]*ripe|over[\s_-]*mature"),
    ("unripe",   r"unripe|un[\s_-]ripe|immature|half[\s_-]*ripe|semi[\s_-]*ripe|turning|breaker|partially"),
    ("unripe",   r"(^|[^a-z])(raw|green)([^a-z]|$)"),
    ("spoiled",  r"rotten|rotted|spoil|decay|stale|(^|[^a-z])bad([^a-z]|$)|_bad|bad_|damaged[\s_-]*rot"),
    ("ripe",     r"fully[\s_-]*ripe|ripened|(^|[^a-z])ripe|ripe$|ripe[^a-z]|mature|fresh|good|ready"),
]
FRUIT_RULES = [("other", r"pineapple|custard[\s_-]*apple|cashew|sugar[\s_-]*apple|rose[\s_-]*apple"),
               ("banana", r"banana"), ("apple", r"apple"), ("mango", r"mango|alphonso"),
               ("tomato", r"tomato"), ("avocado", r"avocado|hass"),
               ("other", r"strawberr|guava|papaya|orange|pomegranate|lime|lemon|grape|pear|peach|kiwi|"
                         r"cucumber|potato|okra|bitter|capsicum|carrot|gooseberry|amla|jujube|lychee|plum")]


def stage_of(text):
    t = text.lower()
    for stage, rx in STAGE_RULES:
        if re.search(rx, t):
            return stage
    return None


def fruit_of(text):
    t = text.lower()
    for fruit, rx in FRUIT_RULES:
        if re.search(rx, t):
            return fruit
    return None


def label_path(rel: str, default_fruit=None, stage_map=None):
    """Stage from the deepest folder/file name that names one; fruit likewise."""
    parts = Path(rel).parts
    stage = None
    for comp in reversed(parts):
        comp_n = Path(comp).stem
        if stage_map:
            for k, v in stage_map.items():
                if re.search(k, comp_n, re.I):
                    stage = v
                    break
        stage = stage or stage_of(comp_n)
        if stage:
            break
    fruit = None
    for comp in reversed(parts):
        fruit = fruit_of(comp)
        if fruit:
            break
    return fruit or default_fruit or "other", stage


def dhash(img: Image.Image) -> str:
    g = np.asarray(img.convert("L").resize((9, 8), Image.BILINEAR), dtype=np.int16)
    bits = (g[:, 1:] > g[:, :-1]).flatten()
    return "%016x" % int("".join("1" if b else "0" for b in bits), 2)


MANIFEST = []                   # dicts: path, fruit, stage, dataset, hash
_counts = Counter()


def add_image(img: Image.Image, ds, fruit, stage):
    if stage in (None, "skip"):
        return False
    key = (ds, fruit, stage)
    if _counts[key] >= PER_CLASS_CAP:
        return False
    if img.mode in ("L", "LA", "I", "I;16", "I;16B", "F", "1", "P") and img.mode != "P":
        return False                      # single-band images (multispectral bands, depth) are not colour photos
    img = img.convert("RGB")
    if min(img.size) < 32:
        return False
    a = np.asarray(img.resize((32, 32)), dtype=np.int16)
    if np.abs(a[..., 0] - a[..., 1]).mean() < 1.5 and np.abs(a[..., 1] - a[..., 2]).mean() < 1.5:
        return False                      # grey image saved as RGB
    img.thumbnail((320, 320))
    h = dhash(img)
    out = W / "unified" / f"{fruit}__{stage}" / f"{ds}_{h}.jpg"
    out.parent.mkdir(parents=True, exist_ok=True)
    img.save(out, quality=92)
    _counts[key] += 1
    MANIFEST.append({"path": str(out), "fruit": fruit, "stage": stage, "dataset": ds, "hash": h})
    return True


def yolo_names(root: Path):
    for y in list(root.rglob("*.yaml")) + list(root.rglob("*.yml")):
        try:
            import yaml
            d = yaml.safe_load(y.read_text(errors="ignore"))
            names = d.get("names") if isinstance(d, dict) else None
            if isinstance(names, dict):
                names = [names[k] for k in sorted(names)]
            if names:
                return [str(n) for n in names]
        except Exception:
            pass
    for c in root.rglob("classes.txt"):
        return [l.strip() for l in c.read_text(errors="ignore").splitlines() if l.strip()]
    return None


def ingest_yolo(ds, root: Path, default_fruit=None, stage_map=None):
    names = yolo_names(root)
    if not names:
        return 0
    n = 0
    for lab in root.rglob("*.txt"):
        if "labels" not in lab.parts or lab.name == "classes.txt":
            continue
        idx = len(lab.parts) - 1 - lab.parts[::-1].index("labels")
        stem_dir = Path(*lab.parts[:idx]) / "images" / Path(*lab.parts[idx + 1:-1])
        img_p = next((stem_dir / (lab.stem + e) for e in [".jpg", ".jpeg", ".png", ".JPG", ".PNG"]
                      if (stem_dir / (lab.stem + e)).exists()), None)
        if img_p is None:
            continue
        try:
            img = Image.open(img_p).convert("RGB")
        except Exception:
            continue
        W_, H_ = img.size
        for line in lab.read_text(errors="ignore").splitlines():
            v = line.split()
            if len(v) < 5:
                continue
            c = int(float(v[0]))
            if c >= len(names):
                continue
            xs, ys = [float(a) for a in v[1::2]], [float(a) for a in v[2::2]]
            if len(v) == 5:   # box: cx cy w h
                cx, cy, bw, bh = map(float, v[1:5])
            else:             # polygon
                cx, cy, bw, bh = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)
            pad = 0.08
            box = (max(0, (cx - bw / 2 - pad * bw) * W_), max(0, (cy - bh / 2 - pad * bh) * H_),
                   min(W_, (cx + bw / 2 + pad * bw) * W_), min(H_, (cy + bh / 2 + pad * bh) * H_))
            fruit, stage = label_path(names[c], default_fruit, stage_map)
            n += add_image(img.crop(tuple(map(int, box))), ds, fruit, stage)
    return n


def ingest_coco(ds, root: Path, default_fruit=None, stage_map=None):
    n = 0
    for js in root.rglob("*.json"):
        try:
            d = json.loads(js.read_text(errors="ignore"))
        except Exception:
            continue
        if not isinstance(d, dict) or "annotations" not in d or "images" not in d:
            continue
        cats = {c["id"]: c["name"] for c in d.get("categories", [])}
        imgs = {i["id"]: i["file_name"] for i in d["images"]}
        by_img = defaultdict(list)
        for a in d["annotations"]:
            by_img[a["image_id"]].append(a)
        for iid, anns in by_img.items():
            fn = imgs.get(iid)
            cand = list(root.rglob(Path(fn).name)) if fn else []
            if not cand:
                continue
            try:
                img = Image.open(cand[0]).convert("RGB")
            except Exception:
                continue
            for a in anns:
                x, y, w, h = a["bbox"]
                fruit, stage = label_path(cats.get(a["category_id"], ""), default_fruit, stage_map)
                n += add_image(img.crop((int(x), int(y), int(x + w), int(y + h))), ds, fruit, stage)
    return n


def ingest_voc(ds, root: Path, default_fruit=None, stage_map=None):
    import xml.etree.ElementTree as ET
    n = 0
    for x in root.rglob("*.xml"):
        try:
            t = ET.parse(x).getroot()
        except Exception:
            continue
        objs = t.findall("object")
        if not objs:
            continue
        fn = (t.findtext("filename") or x.stem)
        cand = [p for p in x.parent.parent.rglob(Path(fn).stem + ".*") if p.suffix.lower() in IMG_EXT][:1]
        if not cand:
            continue
        try:
            img = Image.open(cand[0])
        except Exception:
            continue
        for o in objs:
            b = o.find("bndbox")
            if b is None:
                continue
            box = tuple(int(float(b.findtext(k))) for k in ("xmin", "ymin", "xmax", "ymax"))
            fruit, stage = label_path(o.findtext("name") or "", default_fruit, stage_map)
            n += add_image(img.crop(box), ds, fruit, stage)
    return n


def ingest_folders(ds, root: Path, default_fruit=None, stage_map=None, path_filter=None):
    imgs = [p for p in root.rglob("*") if p.suffix.lower() in IMG_EXT and "/labels/" not in str(p)]
    if path_filter:
        imgs = [p for p in imgs if re.search(path_filter, str(p.relative_to(root)), re.I)]
    random.Random(0).shuffle(imgs)
    n = 0
    for p in imgs:
        fruit, stage = label_path(str(p.relative_to(root)), default_fruit, stage_map)
        if stage in (None, "skip") or _counts[(ds, fruit, stage)] >= PER_CLASS_CAP:
            continue
        try:
            n += add_image(Image.open(p), ds, fruit, stage)
        except Exception:
            pass
    return n


def ingest(ds, root: Path, default_fruit=None, stage_map=None, path_filter=None):
    """Detection datasets are cropped per box; folder datasets are labelled by folder names."""
    n = (ingest_yolo(ds, root, default_fruit, stage_map) + ingest_coco(ds, root, default_fruit, stage_map) +
         ingest_voc(ds, root, default_fruit, stage_map))
    if n == 0:
        n = ingest_folders(ds, root, default_fruit, stage_map, path_filter)
    c = Counter({f"{k[1]}__{k[2]}": v for k, v in _counts.items() if k[0] == ds})
    log(ds, "ok" if n else "no labelled images", dict(c) if n else "folder names had no stage words")
    return n

## Part 1: image datasets (20)

Each entry: name, how to fetch it, default fruit, and any label fix. Labels come from folder names
(`unripe`, `ripe`, `overripe`, `rotten`/`spoiled`, `fresh`, `green`, …) or from detection class names,
and are mapped to RipeSense's four stages. "Fresh" counts as **ripe**; formalin, carbide and "mixed"
folders are skipped. Duplicates across datasets are removed by image hash before the train/val split.

In [ ]:
VISUAL = [
    # name,                 fetch function and arguments,                                                default fruit, stage map
    ("hf_agml_fresh_rotten", lambda: fetch_hf_parquet("hf_agml_fresh_rotten", "Project-AgML/fresh_rotten_fruit_classification", "raw"), None, None),
    ("hf_fruitvision",       lambda: fetch_hf_parquet("hf_fruitvision", "Project-AgML/FruitVision_quality_classification", "raw"), None, None),
    ("hf_densu_fresh_rotten", lambda: fetch_hf_parquet("hf_densu_fresh_rotten", "Densu341/Fresh-rotten-fruit"), None, None),
    ("md_banana_ripeness",   lambda: fetch_mendeley("md_banana_ripeness", "56c75hvm7r", 2, include=r"banana_dataset\.zip"), "banana", None),
    ("md_apple_multispectral", lambda: fetch_mendeley("md_apple_multispectral", "y5h6v8w6ms", 2, include=r"ripeness"), "apple", None),
    ("md_banana_guava",      lambda: fetch_mendeley("md_banana_guava", "56td5w4wz2", 2), None, None),
    ("md_bananaimagebd",     lambda: fetch_mendeley("md_bananaimagebd", "ptfscwtnyz", 2), "banana", None),
    ("md_mango_banana_yolo", lambda: fetch_mendeley("md_mango_banana_yolo", "y3649cmgg6", 3), None, None),
    ("md_strawberry_avocado", lambda: fetch_mendeley("md_strawberry_avocado", "zysvgmxcyz", 1, exclude=r"augment"), None, None),
    ("md_alphonso_mango",    lambda: fetch_mendeley("md_alphonso_mango", "tyghd6gxw2", 1), "mango", None),
    ("md_tomato_maturity",   lambda: fetch_mendeley("md_tomato_maturity", "s42kpg8h37", 1), "tomato", None),
    ("md_spoiled_fresh",     lambda: fetch_mendeley("md_spoiled_fresh", "6ps7gtp2wg", 1), None, None),
    ("md_papaya_maturity",   lambda: fetch_mendeley("md_papaya_maturity", "rcy6y8fhcn", 1), "other", None),
    ("md_apple_stages",      lambda: fetch_mendeley("md_apple_stages", "gfcmdbvw65", 4), "apple", None),
    ("laboro_tomato",        lambda: fetch_url("laboro_tomato", "http://assets.laboro.ai.s3.amazonaws.com/laborotomato/laboro_tomato.zip", "laboro_tomato.zip"), "tomato",
                              {r"fully[_-]?ripened": "ripe", r"half[_-]?ripened": "unripe", r"green": "unripe"}),
    ("kg_fresh_rotten",      lambda: fetch_kaggle("kg_fresh_rotten", "sriramr/fruits-fresh-and-rotten-for-classification"), None, None),
    ("kg_fruitnet_quality",  lambda: fetch_kaggle("kg_fruitnet_quality", "shashwatwork/fruitnet-indian-fruits-dataset-with-quality"), None, None),
    ("kg_fruit_ripeness",    lambda: fetch_kaggle("kg_fruit_ripeness", "asadullahprl/fruits-ripeness-classification-dataset"), None, None),
    ("kg_fresh_stale",       lambda: fetch_kaggle("kg_fresh_stale", "swoyam2609/fresh-and-stale-images-of-fruits-and-vegetables"), None, None),
    ("fs_trimodal_rgb",      lambda: fetch_remotezip("fs_trimodal_rgb", "https://ndownloader.figshare.com/files/60094544",
                                                     r"classified/.*\.(jpe?g|png)$", max_members=PER_CLASS_CAP,
                                                     exclude_re=r"ir[\s_-]*fusion|thermal|(^|/)ir(/|_)"), None, None),
]

for name, fetch, fruit, smap in VISUAL:
    t0 = time.time()
    try:
        root = fetch()
        ingest(name, Path(root), fruit, smap)
    except Exception as e:
        log(name, "FAILED", f"{type(e).__name__}: {e}")
    print(f"   ({time.time() - t0:.0f}s)")

In [ ]:
# ── Merge: drop duplicates, drop tiny classes, split train/val ────────────────
import pandas as pd

man = pd.DataFrame(MANIFEST)
print("images collected:", len(man))
before = len(man)
man = man.drop_duplicates("hash")
print(f"removed {before - len(man)} duplicates across datasets")
man["label"] = man["fruit"] + "__" + man["stage"]
counts = man["label"].value_counts()
small = counts[counts < MIN_CLASS].index.tolist()
if small:
    print("dropping small classes:", small)
man = man[~man["label"].isin(small)].reset_index(drop=True)
rng = np.random.default_rng(0)
man["split"] = "train"
for lab, g in man.groupby("label"):
    val_idx = rng.choice(g.index, size=max(5, int(0.15 * len(g))), replace=False)
    man.loc[val_idx, "split"] = "val"
LABELS = sorted(man["label"].unique())
man.to_csv(W / "manifest.csv", index=False)
print(pd.crosstab(man["label"], man["split"]))
print("\nper dataset:\n", man["dataset"].value_counts())

## Train the image model
MobileNetV3-Large, ImageNet weights, input 224×224. Preprocessing at validation time is a plain resize to
224×224 + ImageNet normalisation, exactly what `ripesense/features.py` does on the Pi.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms

dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)
norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.55, 1.0), ratio=(0.8, 1.25)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(0.2), transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.25, saturation=0.15, hue=0.015),  # small hue: colour IS the label
    transforms.ToTensor(), norm])
val_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), norm])


class DS(Dataset):
    def __init__(self, df, tf):
        self.p, self.y, self.tf = df["path"].tolist(), [LABELS.index(l) for l in df["label"]], tf

    def __len__(self):
        return len(self.p)

    def __getitem__(self, i):
        return self.tf(Image.open(self.p[i]).convert("RGB")), self.y[i]


tr, va = man[man.split == "train"], man[man.split == "val"]
freq = tr["label"].value_counts()
weights = tr["label"].map(lambda l: 1.0 / freq[l]).values
sampler = WeightedRandomSampler(torch.tensor(weights, dtype=torch.double), num_samples=len(tr), replacement=True)
tl = DataLoader(DS(tr, train_tf), batch_size=BATCH, sampler=sampler, num_workers=2, pin_memory=True)
vl = DataLoader(DS(va, val_tf), batch_size=BATCH, num_workers=2)

net = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, len(LABELS))
net.to(dev)
opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=EPOCHS * len(tl), pct_start=0.15)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)
scaler = torch.amp.GradScaler(enabled=dev == "cuda")


def evaluate():
    net.eval()
    P, Y = [], []
    with torch.no_grad():
        for x, y in vl:
            with torch.autocast(dev, enabled=dev == "cuda"):
                P.append(net(x.to(dev)).float().softmax(1).cpu())
            Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()


best, best_state = -1, None
for ep in range(EPOCHS):
    net.train()
    t0 = time.time()
    for x, y in tl:
        x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(dev, enabled=dev == "cuda"):
            loss = loss_fn(net(x), y)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); sched.step()
    P, Y = evaluate()
    acc = float((P.argmax(1) == Y).mean())
    st = np.array([l.split("__")[1] for l in LABELS])
    stage_acc = float((st[P.argmax(1)] == st[Y]).mean())
    print(f"epoch {ep + 1}/{EPOCHS}  val acc {acc:.3f}  stage acc {stage_acc:.3f}  ({time.time() - t0:.0f}s)")
    if stage_acc > best:
        best, best_state = stage_acc, {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
net.load_state_dict(best_state)

In [ ]:
# ── Report: per class, per stage, per dataset ──────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix

P, Y = evaluate()
pred = P.argmax(1)
print(classification_report(Y, pred, labels=range(len(LABELS)), target_names=LABELS, zero_division=0))
STAGES = ["unripe", "ripe", "overripe", "spoiled"]
st_true = [LABELS[i].split("__")[1] for i in Y]
st_pred = [LABELS[i].split("__")[1] for i in pred]
print("stage confusion (rows = truth):", STAGES)
print(confusion_matrix(st_true, st_pred, labels=STAGES))
va = va.assign(pred=[LABELS[i] for i in pred])
per_ds = va.groupby("dataset").apply(lambda g: float((g["pred"].str.split("__").str[1] == g["stage"]).mean()))
print("\nstage accuracy per source dataset:\n", per_ds.round(3))
report = {"labels": LABELS, "val_images": int(len(va)), "train_images": int(len(tr)),
          "val_accuracy": float((pred == Y).mean()),
          "stage_accuracy": float(np.mean(np.array(st_true) == np.array(st_pred))),
          "per_dataset_stage_accuracy": per_ds.round(3).to_dict(),
          "classification_report": classification_report(Y, pred, labels=range(len(LABELS)), target_names=LABELS,
                                                         zero_division=0, output_dict=True),
          "sources": [r for r in LOG]}

In [ ]:
# ── Export ONNX in the format the Pi app loads ─────────────────────────────────
OUT = W / "ripesense_models"; (OUT / "models").mkdir(parents=True, exist_ok=True)
onnx_path = OUT / "models" / "fruit_classifier.onnx"
net.cpu().eval()
torch.onnx.export(net, torch.randn(1, 3, 224, 224), str(onnx_path), input_names=["input"], output_names=["logits"],
                  dynamic_axes={"input": {0: "n"}, "logits": {0: "n"}}, opset_version=17, dynamo=False)
Path(str(onnx_path) + ".labels.json").write_text(json.dumps(LABELS))

import onnxruntime as ort
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
xb, _ = next(iter(vl))
diff = np.abs(sess.run(None, {"input": xb[:8].numpy()})[0] - net(xb[:8]).detach().numpy()).max()
print(f"ONNX matches PyTorch (max diff {diff:.2e}); size {onnx_path.stat().st_size / 1e6:.1f} MB")
t0 = time.time(); [sess.run(None, {"input": xb[:1].numpy()}) for _ in range(20)]
print(f"CPU latency here: {(time.time() - t0) / 20 * 1000:.0f} ms/image (a Pi 5 is roughly 2–4x slower)")

## Part 2: gas-sensor datasets (10)

These don't match your exact sensor, chamber or fruit amount, so they can't be used raw. What the
notebook does with them:
- **Fruit/food datasets with labels** (Zenodo e-nose, TriModal MQ-4) → windows turned into the same
  features RipeSense computes (`g_rise`, `g_slope`, temperature, humidity) and saved as
  `public_gas_rows.csv`. On the Pi, `ripesense train --public models/public_gas_rows.csv` mixes them in at
  low weight with your own labelled scans.
- **Ethylene calibration datasets** (UCI drift) → how a metal-oxide sensor's response grows with
  ethylene concentration (`ethylene_curve.json`).
- **Context datasets** (UCI banana/wine, wine spoilage, cold-storage CO2) → summary stats only; their
  labels aren't fruit ripeness stages.

In [ ]:
GAS_ROWS = []   # source, fruit_type, stage, g_rise, g_slope, g_temp, g_humidity


def show_tree(root: Path, n=25):
    files = [p for p in Path(root).rglob("*") if p.is_file()]
    print(f"   {len(files)} files, e.g.:")
    for p in files[:n]:
        print("    ", p.relative_to(root), f"{p.stat().st_size / 1e3:.0f} kB")
    return files


def read_table(p: Path):
    for kw in ({"sep": None, "engine": "python"}, {"sep": r"\s+", "engine": "python"}, {"sep": ","}):
        try:
            df = pd.read_csv(p, **kw)
            if df.shape[1] >= 2:
                return df
        except Exception:
            pass
    return None


def window_rows(source, fruit, stage, sig, t=None, temp=None, hum=None, rises=True, win=120):
    """Turn a long sensor trace into scan-like windows: baseline = first sample of each window's trace."""
    sig = np.asarray(sig, float)
    ok = np.isfinite(sig) & (sig > 0)
    sig = sig[ok]
    if len(sig) < 20:
        return 0
    base = np.median(sig[: max(5, len(sig) // 20)])
    n = 0
    for s in range(0, len(sig) - win // 4, max(win // 2, 1)):
        w = sig[s:s + win]
        if len(w) < 10:
            continue
        r = w / base if rises else base / w
        rise = float(np.median(r[-5:]) - 1)
        slope = float(np.polyfit(np.arange(len(r)) / 60.0, r, 1)[0])
        GAS_ROWS.append({"source": source, "fruit_type": fruit, "stage": stage, "g_rise": rise, "g_slope": slope,
                         "g_temp": float(np.nanmedian(temp)) if temp is not None else np.nan,
                         "g_humidity": float(np.nanmedian(hum)) if hum is not None else np.nan})
        n += 1
    return n

In [ ]:
# G1: Zenodo "Food Freshness Electronic Nose Dataset" (9 MQ sensors, readings every 24 h)
try:
    root = fetch_url("gas_zenodo_enose", "https://zenodo.org/records/17285312/files/Food-Freshness-E-Nose-Dataset.zip?download=1",
                     "Food-Freshness-E-Nose-Dataset.zip")
    files = show_tree(root)
    n = 0
    for p in files:
        if p.suffix.lower() not in (".csv", ".txt", ".xlsx", ".xls"):
            continue
        df = pd.read_excel(p) if p.suffix.lower().startswith(".xls") else read_table(p)
        if df is None:
            continue
        cols = {c.lower(): c for c in df.columns}
        mq = [c for c in df.columns if re.search(r"mq\s*-?\d", str(c), re.I)]
        labcol = next((cols[k] for k in cols if re.search(r"label|class|status|fresh|quality|condition", k)), None)
        fruit = fruit_of(str(p.relative_to(root))) or "other"
        if mq and labcol is not None:
            for lab, g in df.groupby(labcol):
                stage = stage_of(str(lab))
                if stage in (None, "skip"):
                    continue
                sig = g[mq].apply(pd.to_numeric, errors="coerce").mean(axis=1).values
                n += window_rows("zenodo_enose", fruit, stage, sig, win=min(60, max(10, len(sig) // 4)))
        elif mq:
            stage = stage_of(str(p.relative_to(root)))
            if stage not in (None, "skip"):
                sig = df[mq].apply(pd.to_numeric, errors="coerce").mean(axis=1).values
                n += window_rows("zenodo_enose", fruit, stage, sig)
        if n == 0 and p == files[0]:
            print("   columns:", list(df.columns)[:20])
    log("gas_zenodo_enose", "ok" if n else "downloaded, no stage labels found", f"{n} rows")
except Exception as e:
    log("gas_zenodo_enose", "FAILED", e)

In [ ]:
# G2: TriModal Ripeness 6 (Figshare): MQ-4 gas logs per fruit, Spoiled vs Not_spoiled. Range requests only.
try:
    root = fetch_remotezip("gas_trimodal", "https://ndownloader.figshare.com/files/60094544", r"\.txt$")
    n = 0
    for p in root.rglob("*.txt"):
        rel = str(p.relative_to(root))
        fruit, stage = label_path(rel)
        if stage in (None, "skip"):
            continue
        df = read_table(p)
        if df is None:
            continue
        num = df.apply(pd.to_numeric, errors="coerce")
        adc = next((c for c in num.columns if re.search(r"adc|raw|analog", str(c), re.I)), None)
        res = next((c for c in num.columns if re.search(r"res|rs", str(c), re.I)), None)
        col, rises = (adc, True) if adc is not None else (res, False) if res is not None else (num.columns[-1], True)
        n += window_rows("figshare_trimodal_mq4", fruit, stage, num[col].values, rises=rises)
    log("gas_trimodal", "ok" if n else "no rows", f"{n} rows")
except Exception as e:
    log("gas_trimodal", "FAILED", e)

In [ ]:
# G3: UCI 362 "Gas sensors for home activity monitoring": 8 MOX sensors + T/RH, banana vs wine vs background.
def uci_zip(uid):
    html = S.get(f"https://archive.ics.uci.edu/dataset/{uid}", timeout=30).text
    m = re.search(rf"/static/public/{uid}/[^\"']+\.zip", html)
    if not m:
        raise RuntimeError("download link not found on UCI page")
    return "https://archive.ics.uci.edu" + m.group(0)


try:
    root = fetch_url("gas_uci362", uci_zip(362), "uci362.zip")
    meta_p = next(root.rglob("HT_Sensor_metadata.dat")); data_p = next(root.rglob("HT_Sensor_dataset.dat"))
    meta = pd.read_csv(meta_p, sep=r"\s+", engine="python")
    data = pd.read_csv(data_p, sep=r"\s+", engine="python")
    data.columns = ["id", "time", "R1", "R2", "R3", "R4", "R5", "R6", "R7", "R8", "Temp", "Humidity"][:data.shape[1]]
    stats = []
    by_id = dict(tuple(data.groupby("id")))
    for _, m in meta.iterrows():
        d = by_id.get(m["id"])
        if d is None:
            continue
        pre, dur = d[d.time < 0], d[(d.time >= 0) & (d.time <= m["dt"])]
        if len(pre) < 5 or len(dur) < 5:
            continue
        R = ["R1", "R2", "R3", "R4", "R5", "R6", "R7", "R8"]
        resp = (pre[R].median() / dur[R].median()).mean() - 1      # MOX: resistance falls with gas
        stats.append({"class": m["class"], "rise": resp, "temp": dur.Temp.median(), "hum": dur.Humidity.median()})
        if m["class"] == "banana":   # bananas on a kitchen counter: treat as ripe fruit
            GAS_ROWS.append({"source": "uci362_banana", "fruit_type": "banana", "stage": "ripe", "g_rise": float(resp),
                             "g_slope": np.nan, "g_temp": float(dur.Temp.median()), "g_humidity": float(dur.Humidity.median())})
    st = pd.DataFrame(stats)
    print(st.groupby("class")["rise"].describe().round(3))
    log("gas_uci362", "ok", f"{(st['class'] == 'banana').sum()} banana rows")
except Exception as e:
    log("gas_uci362", "FAILED", e)

In [ ]:
# G4 + G5: UCI 224 and 270 "Gas sensor array drift": 16 MOX sensors x 6 gases. Gas 2 = ethylene.
# Fit response = a * ppm^b for ethylene, the classic metal-oxide power law.
ETH = []
for uid in (224, 270):
    try:
        root = fetch_url(f"gas_uci{uid}", uci_zip(uid), f"uci{uid}.zip")
        for p in root.rglob("*.dat"):
            for line in p.read_text(errors="ignore").splitlines():
                head, *feats = line.split()
                if ";" not in head:
                    continue
                gas_id, conc = head.split(";")
                if int(float(gas_id)) != 2 or float(conc) <= 0:
                    continue
                f = dict(x.split(":") for x in feats if ":" in x)
                norm_dr = [abs(float(f.get(str(8 * s + 2), "nan"))) for s in range(16)]   # normalised ΔR per sensor
                ETH.append((float(conc), float(np.nanmedian(norm_dr))))
        log(f"gas_uci{uid}", "ok", f"{len(ETH)} ethylene exposures so far")
    except Exception as e:
        log(f"gas_uci{uid}", "FAILED", e)
eth_curve = {}
if len(ETH) > 20:
    c, r = np.array(ETH).T
    ok = (c > 0) & (r > 0)
    b, loga = np.polyfit(np.log(c[ok]), np.log(r[ok]), 1)
    eth_curve = {"law": "response = a * ppm**b", "a": float(np.exp(loga)), "b": float(b),
                 "ppm_range": [float(c.min()), float(c.max())], "n": int(ok.sum()),
                 "note": "UCI gas sensor drift (Vergara 2012), median of 16 MOX sensors. Use b to linearise g_rise."}
    print("ethylene power law:", eth_curve)

In [ ]:
# G6–G9 (optional, large): UCI ethylene time series (not fruit, but how MOX sensors react to ethylene).
# 309 turbulent mixtures, 322 dynamic mixtures, 251 open sampling, 361 twin arrays.
if ETHYLENE_EXTRAS:
    for uid in (309, 322, 251, 361):
        try:
            root = fetch_url(f"gas_uci{uid}", uci_zip(uid), f"uci{uid}.zip")
            show_tree(root, 8)
            log(f"gas_uci{uid}", "downloaded", "raw time series kept for your own exploration")
        except Exception as e:
            log(f"gas_uci{uid}", "FAILED", e)
else:
    for uid in (309, 322, 251, 361):
        log(f"gas_uci{uid}", "skipped", "set ETHYLENE_EXTRAS = True to download")

In [ ]:
# G10: wine spoilage e-nose (Mendeley vpc887d53s, acetic-acid thresholds): spoilage VOCs, summary only.
try:
    root = fetch_mendeley("gas_wine_spoilage", "vpc887d53s", 3)
    files = show_tree(root, 10)
    log("gas_wine_spoilage", "ok", f"{len(files)} files (context only: wine, not fruit)")
except Exception as e:
    log("gas_wine_spoilage", "FAILED", e)

# G11: IoT cold storage (Mendeley czz68d9fwj): fruit, temp, humidity, light, CO2 ppm, Good/Bad conditions.
try:
    root = fetch_mendeley("gas_coldstore_co2", "czz68d9fwj", 1)
    for p in list(root.rglob("*.csv")) + list(root.rglob("*.xlsx")):
        df = pd.read_excel(p) if p.suffix == ".xlsx" else pd.read_csv(p)
        print(df.head())
        co2 = next((c for c in df.columns if "co2" in c.lower()), None)
        cls = next((c for c in df.columns if c.lower() in ("class", "label")), None)
        if co2 and cls:
            print(df.groupby(cls)[co2].describe().round(0))
    log("gas_coldstore_co2", "ok", "context only: labels describe storage conditions, not ripeness")
except Exception as e:
    log("gas_coldstore_co2", "FAILED", e)

In [ ]:
gas = pd.DataFrame(GAS_ROWS)
if len(gas):
    gas = gas.replace([np.inf, -np.inf], np.nan).dropna(subset=["g_rise"])
    gas = gas[(gas.g_rise > -0.9) & (gas.g_rise < 50)]
    gas.to_csv(OUT / "models" / "public_gas_rows.csv", index=False)
    print(gas.groupby(["source", "stage"]).g_rise.describe().round(3))
if eth_curve:
    (OUT / "models" / "ethylene_curve.json").write_text(json.dumps(eth_curve, indent=1))

## Part 3: O2 and respiration

There are no public datasets of raw O2-sensor logs from fruit, so this part uses published measurements
instead. It also works out how far O2 can move inside the RipeSense chamber during one scan.

In [ ]:
RESP = pd.DataFrame([
    # fruit, condition, temp_C, O2 uptake (mL/kg/h), CO2 output (mL/kg/h), ethylene (uL/kg/h), source
    # nmol/kg/s -> mL/kg/h: x 3600 s x 24.06 mL/umol-scale at 20 C  (99 nmol/kg/s = 8.6 mL/kg/h)
    ("strawberry", "air (21% O2)", 10, 20.2, 27.0, None, "Sensors 2020, 20(12):3589, doi:10.3390/s20123589"),
    ("strawberry", "2% O2", 10, 10.5, None, None, "Sensors 2020, 20(12):3589, doi:10.3390/s20123589"),
    ("apple (Pinova)", "CA 2% O2, 1.3% CO2, start", 2, 2.2, 4.2, None, "Sensors 2020, 20(12):3589"),
    ("apple (Pinova)", "CA 2% O2, 1.3% CO2, steady", 2, 1.6, 2.7, None, "Sensors 2020, 20(12):3589"),
    ("apple (Granny Smith)", "air, midpoint of range", 0.5, None, 3.0, 3.5, "UC Davis Produce Facts: 2-4 mL CO2, 1-6 uL C2H4"),
    ("apple (Jonagored)", "optimal harvest, day 0", 20, None, 8.6, 0.0, "J. Agroindustrial (UGM) 25031: 99 nmol/kg/s"),
    ("apple (Jonagored)", "late harvest, day 0", 20, None, 13.3, 0.0, "J. Agroindustrial (UGM) 25031: 153 nmol/kg/s"),
    ("apple (Jonagored)", "late harvest, day 7", 20, None, 11.2, 6.9, "J. Agroindustrial (UGM) 25031: 129 nmol/kg/s; C2H4 units unclear"),
    ("apple (class)", "climacteric, midpoint of class", 20, None, None, 55.0, "UF/IFAS postharvest: apples 10-100 uL/kg/h"),
], columns=["fruit", "condition", "temp_c", "o2_ml_kg_h", "co2_ml_kg_h", "ethylene_ul_kg_h", "source"])
RESP.to_csv(OUT / "models" / "respiration_reference.csv", index=False)
print(RESP.to_string(index=False))

# What one scan can see. Chamber 185 mm tall, ~128 x 128 mm inside -> ~3.0 L of air minus the fruit.
chamber_l, fruit_kg, scan_s = 3.0, 0.2, 120
for label, rate in [("apple, cold-stored (2 C)", 2.0), ("apple, room temp, preclimacteric", 8.0),
                    ("apple, ripe at 20 C (climacteric)", 20.0)]:
    ml_per_scan = rate * fruit_kg * scan_s / 3600
    o2_drop_pct = ml_per_scan / (chamber_l * 1000) * 100
    co2_ppm = ml_per_scan / (chamber_l * 1000) * 1e6
    print(f"{label:38s}  O2 drop {o2_drop_pct:.4f} %-pts   CO2 rise {co2_ppm:5.0f} ppm   per {scan_s}s scan")
print("\nA typical electrochemical O2 sensor resolves about 0.1 %-pts, so a sealed chamber needs roughly"
      "\n30-60 minutes per 200 g of ripe fruit before O2 moves enough to read. Use O2 for long box tests;"
      "\nfor a 2-minute scan, the MQ/BME gas rise (and a CO2 sensor, if you have one) carries the signal.")

In [ ]:
# ── Package for the Pi ─────────────────────────────────────────────────────────
(OUT / "reports").mkdir(exist_ok=True)
report["gas_rows"] = int(len(gas)) if len(gas) else 0
report["ethylene_curve"] = eth_curve
(OUT / "reports" / "training_report.json").write_text(json.dumps(report, indent=1, default=str))
pd.DataFrame(LOG, columns=["dataset", "status", "detail"]).to_csv(OUT / "reports" / "sources.csv", index=False)
man.groupby(["dataset", "label"]).size().rename("images").reset_index().to_csv(OUT / "reports" / "images_per_dataset.csv", index=False)
(OUT / "README.txt").write_text(
    "Copy the models/ files into ~/ripesense/models/ on the Pi:\n"
    "  fruit_classifier.onnx + fruit_classifier.onnx.labels.json  -> loaded automatically by the scanner\n"
    "  public_gas_rows.csv  -> ripesense train --public models/public_gas_rows.csv  (after labelling scans)\n"
    "  ethylene_curve.json, respiration_reference.csv  -> reference data\n"
    "reports/ has accuracy per class and per source dataset.\n")
shutil.make_archive(str(W / "ripesense_models"), "zip", str(OUT))
shutil.copy(W / "ripesense_models.zip", "/content/ripesense_models.zip") if Path("/content").exists() else None
print(pd.DataFrame(LOG, columns=["dataset", "status", "detail"]).to_string(index=False))
print("\nDone. Download /content/ripesense_models.zip from the Files panel (left).")
try:
    from google.colab import files
    files.download("/content/ripesense_models.zip")
except Exception:
    pass